# 24 · Boundary Failure Geometry

This notebook follows Notebooks 22 and 23.

Those notebooks showed:
- zeta vs Poisson is easy
- GUE vs Poisson is easy
- zeta vs GUE is hard
- linear and nonlinear boundaries do not materially improve zeta-vs-GUE separation

Now we ask a more detailed question:

> What does boundary failure actually look like geometrically?

## Main diagnostics

For each binary task, we study:
1. classifier confidence distribution
2. correctness vs confidence
3. boundary margin distribution
4. local decision surface slices
5. simple overlap metrics in decision space

The main contrast is:

- **zeta vs GUE** → suspected intrinsic overlap
- **zeta vs Poisson / GUE vs Poisson** → expected clean separation

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Minimal feature set

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def build_feature_matrix(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [window_entropy(w), unevenness(w), ratio_mean(w)]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## Train / test split

In [ ]:
zeta_base = zeta_gaps_full[:700]
poisson_base = poisson_gaps_full[:700]
gue_base = gue_gaps_full[:950]

_, zeta_X = build_feature_matrix(zeta_base, k=5)
_, poisson_X = build_feature_matrix(poisson_base, k=5)
_, gue_X = build_feature_matrix(gue_base, k=5)

m = min(len(zeta_X), len(poisson_X), len(gue_X))
zeta_X = zeta_X[:m]
poisson_X = poisson_X[:m]
gue_X = gue_X[:m]

def split_train_test(X, frac=0.6):
    n = len(X)
    cut = int(frac * n)
    return X[:cut], X[cut:]

zeta_train, zeta_test = split_train_test(zeta_X, frac=0.6)
poisson_train, poisson_test = split_train_test(poisson_X, frac=0.6)
gue_train, gue_test = split_train_test(gue_X, frac=0.6)

zeta_train.shape, zeta_test.shape

## Helpers: standardization, PCA, logistic, RBF lift

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

def fit_pca_2d(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1.0
    Xs = (X - mean) / std
    Xc = Xs - Xs.mean(axis=0)
    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    comps = eigvecs[:, :2]
    Z = Xc @ comps
    explained = eigvals[:2] / eigvals.sum()
    return {"mean": mean, "std": std, "comps": comps, "explained": explained}

def transform_pca_2d(X, pca_model):
    Xs = (X - pca_model["mean"]) / pca_model["std"]
    Xc = Xs - Xs.mean(axis=0)
    return Xc @ pca_model["comps"]

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def predict_logistic(X, w):
    return (predict_proba_logistic(X, w) >= 0.5).astype(int)

def quadratic_features(X):
    x1, x2, x3 = X[:,0], X[:,1], X[:,2]
    return np.column_stack([x1, x2, x3, x1*x1, x2*x2, x3*x3, x1*x2, x1*x3, x2*x3])

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

## Task runner with confidence diagnostics

We use the RBF prototype boundary, since Notebook 23 showed it is at least as expressive as the linear and quadratic versions.

In [ ]:
def run_failure_geometry_task(name, X0_train, X1_train, X0_test, X1_test):
    m_train = min(len(X0_train), len(X1_train))
    m_test = min(len(X0_test), len(X1_test))

    X_train = np.vstack([X0_train[:m_train], X1_train[:m_train]])
    y_train = np.array([0] * m_train + [1] * m_train)

    X_test = np.vstack([X0_test[:m_test], X1_test[:m_test]])
    y_test = np.array([0] * m_test + [1] * m_test)

    Xtr_std, Xte_std, mean, std = standardize_train_test(X_train, X_test)

    prototypes = choose_prototypes(Xtr_std, n_proto=20)
    gamma = estimate_rbf_gamma(Xtr_std)

    R_train = rbf_features(Xtr_std, prototypes, gamma)
    R_test = rbf_features(Xte_std, prototypes, gamma)

    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    scores = decision_function_logistic(R_test, w)
    probs = predict_proba_logistic(R_test, w)
    preds = (probs >= 0.5).astype(int)
    acc = accuracy(y_test, preds)

    pca_model = fit_pca_2d(X_train)
    Z_test = transform_pca_2d(X_test, pca_model)

    conf = np.abs(probs - 0.5) * 2.0
    correct = (preds == y_test).astype(int)

    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]
    conf0 = conf[y_test == 0]
    conf1 = conf[y_test == 1]
    correct_conf = conf[correct == 1]
    wrong_conf = conf[correct == 0]

    normw = np.linalg.norm(w[1:]) if np.linalg.norm(w[1:]) > 1e-12 else 1.0
    dists = np.abs(scores) / normw
    dists0 = dists[y_test == 0]
    dists1 = dists[y_test == 1]

    overlap_scores = overlap_coefficient_from_hist(scores0, scores1, bins=30)
    overlap_dists = overlap_coefficient_from_hist(dists0, dists1, bins=30)

    return {
        "name": name,
        "accuracy": acc,
        "Z_test": Z_test,
        "y_test": y_test,
        "scores": scores,
        "probs": probs,
        "preds": preds,
        "confidence": conf,
        "correct": correct,
        "scores0": scores0,
        "scores1": scores1,
        "conf0": conf0,
        "conf1": conf1,
        "correct_conf": correct_conf,
        "wrong_conf": wrong_conf,
        "dists0": dists0,
        "dists1": dists1,
        "overlap_scores": overlap_scores,
        "overlap_dists": overlap_dists,
        "pca_model": pca_model,
        "mean_std": (mean, std),
        "prototypes": prototypes,
        "gamma": gamma,
        "weights": w,
    }

## Run the three main tasks

In [ ]:
task_zeta_gue = run_failure_geometry_task("zeta vs GUE", zeta_train, gue_train, zeta_test, gue_test)
task_zeta_pois = run_failure_geometry_task("zeta vs Poisson", zeta_train, poisson_train, zeta_test, poisson_test)
task_gue_pois = run_failure_geometry_task("GUE vs Poisson", gue_train, poisson_train, gue_test, poisson_test)

task_zeta_gue["accuracy"], task_zeta_pois["accuracy"], task_gue_pois["accuracy"]

## Confidence-vs-correctness histograms

If failure is intrinsic overlap, wrong predictions should tend to have low confidence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, task in zip(axes, [task_zeta_gue, task_zeta_pois, task_gue_pois]):
    ax.hist(task["correct_conf"], bins=25, alpha=0.6, density=True, label="correct")
    ax.hist(task["wrong_conf"], bins=25, alpha=0.6, density=True, label="wrong")
    ax.set_title(f"{task['name']} · acc={task['accuracy']:.3f}")
    ax.set_xlabel("confidence")

axes[0].set_ylabel("density")
axes[0].legend()
plt.tight_layout()
plt.show()

## Decision-score overlap

Heavy overlap indicates a real boundary-failure regime.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, task, l0, l1 in [
    (axes[0], task_zeta_gue, "zeta", "GUE"),
    (axes[1], task_zeta_pois, "zeta", "Poisson"),
    (axes[2], task_gue_pois, "GUE", "Poisson"),
]:
    ax.hist(task["scores0"], bins=25, alpha=0.6, density=True, label=l0)
    ax.hist(task["scores1"], bins=25, alpha=0.6, density=True, label=l1)
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.set_title(f"{task['name']} · score overlap={task['overlap_scores']:.3f}")
    ax.set_xlabel("decision score")

axes[0].set_ylabel("density")
axes[0].legend()
plt.tight_layout()
plt.show()

## Distance-to-boundary overlap

If both classes pile up near the boundary, separation is intrinsically weak.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, task, l0, l1 in [
    (axes[0], task_zeta_gue, "zeta", "GUE"),
    (axes[1], task_zeta_pois, "zeta", "Poisson"),
    (axes[2], task_gue_pois, "GUE", "Poisson"),
]:
    ax.hist(task["dists0"], bins=25, alpha=0.6, density=True, label=l0)
    ax.hist(task["dists1"], bins=25, alpha=0.6, density=True, label=l1)
    ax.set_title(f"{task['name']} · dist overlap={task['overlap_dists']:.3f}")
    ax.set_xlabel("distance to boundary")

axes[0].set_ylabel("density")
axes[0].legend()
plt.tight_layout()
plt.show()

## PCA slices colored by confidence

This gives a direct geometric view of where the classifier is uncertain.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

for ax, task in zip(axes, [task_zeta_gue, task_zeta_pois, task_gue_pois]):
    Z = task["Z_test"]
    conf = task["confidence"]
    y = task["y_test"]
    sc = ax.scatter(Z[:,0], Z[:,1], c=conf, s=16, alpha=0.65)
    ax.set_title(task["name"])
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.85, label="classifier confidence")
plt.tight_layout()
plt.show()

## Local decision-surface slice

We fix the third feature (ratio_mean) at its test-set median and vary the first two features:
- entropy
- unevenness

This is only a slice, but it helps visualize how smooth or unstable the decision geometry is.

In [ ]:
def evaluate_rbf_surface(task, x1_range, x2_range, fixed_ratio):
    mean, std = task["mean_std"]
    prototypes = task["prototypes"]
    gamma = task["gamma"]
    w = task["weights"]

    xx, yy = np.meshgrid(x1_range, x2_range)
    grid = np.column_stack([xx.ravel(), yy.ravel(), np.full(xx.size, fixed_ratio)])

    grid_std = (grid - mean) / std
    R_grid = rbf_features(grid_std, prototypes, gamma)
    probs = predict_proba_logistic(R_grid, w).reshape(xx.shape)
    return xx, yy, probs

def surface_plot(ax, task, X_test_ref, title):
    ent = X_test_ref[:,0]
    unev = X_test_ref[:,1]
    ratio_med = np.median(X_test_ref[:,2])

    x1_range = np.linspace(ent.min() - 0.2, ent.max() + 0.2, 120)
    x2_range = np.linspace(unev.min() - 0.2, unev.max() + 0.2, 120)

    xx, yy, probs = evaluate_rbf_surface(task, x1_range, x2_range, ratio_med)
    im = ax.contourf(xx, yy, probs, levels=20)
    ax.set_title(title)
    ax.set_xlabel("entropy")
    ax.set_ylabel("unevenness")
    return im

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

X_test_zg = np.vstack([zeta_test[:len(gue_test)], gue_test[:len(zeta_test)]])
X_test_zp = np.vstack([zeta_test[:len(poisson_test)], poisson_test[:len(zeta_test)]])
X_test_gp = np.vstack([gue_test[:len(poisson_test)], poisson_test[:len(gue_test)]])

im = surface_plot(axes[0], task_zeta_gue, X_test_zg, "zeta vs GUE")
im = surface_plot(axes[1], task_zeta_pois, X_test_zp, "zeta vs Poisson")
im = surface_plot(axes[2], task_gue_pois, X_test_gp, "GUE vs Poisson")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="predicted class-1 probability")
plt.tight_layout()
plt.show()

## Overlap summary table

In [ ]:
overlap_summary = {
    "zeta_vs_GUE": {
        "accuracy": task_zeta_gue["accuracy"],
        "score_overlap": task_zeta_gue["overlap_scores"],
        "distance_overlap": task_zeta_gue["overlap_dists"],
        "mean_confidence": float(np.mean(task_zeta_gue["confidence"])),
    },
    "zeta_vs_Poisson": {
        "accuracy": task_zeta_pois["accuracy"],
        "score_overlap": task_zeta_pois["overlap_scores"],
        "distance_overlap": task_zeta_pois["overlap_dists"],
        "mean_confidence": float(np.mean(task_zeta_pois["confidence"])),
    },
    "GUE_vs_Poisson": {
        "accuracy": task_gue_pois["accuracy"],
        "score_overlap": task_gue_pois["overlap_scores"],
        "distance_overlap": task_gue_pois["overlap_dists"],
        "mean_confidence": float(np.mean(task_gue_pois["confidence"])),
    },
}
overlap_summary

## Compact summary

In [ ]:
summary = {
    "overlap_summary": overlap_summary,
    "tasks": {
        "zeta_vs_GUE": {
            "accuracy": task_zeta_gue["accuracy"],
        },
        "zeta_vs_Poisson": {
            "accuracy": task_zeta_pois["accuracy"],
        },
        "GUE_vs_Poisson": {
            "accuracy": task_gue_pois["accuracy"],
        },
    },
}
summary

## Notes

- If zeta-vs-GUE has both high overlap and low confidence while the Poisson tasks have lower overlap and higher confidence, then boundary failure is structural rather than a simple model miss.
- Confidence slices and decision-surface slices are only diagnostics, but they make the failure regime visible instead of reducing it to a single accuracy number.
- This notebook uses the RBF prototype classifier because it was the strongest nonlinear family from Notebook 23.

## Next directions

A natural next notebook could do one of these:

1. bootstrap these overlap metrics  
2. compare failure geometry for several feature sets  
3. move to longer windows to see whether failure persists nonlocally  
4. evaluate GUE-trained confidence directly on zeta-vs-GUE mixtures